### Read Bronze Payments

In [0]:
from pyspark.sql import functions as F

BRONZE_TABLE = "olist_mlops.bronze.payments"
SILVER_TABLE = "olist_mlops.silver.payments_clean"

bronze_payments = spark.table(BRONZE_TABLE)

print("Bronze rows:", bronze_payments.count())

bronze_payments.printSchema()

display(bronze_payments.limit(5))

### Validate values before type conversion
- Check type conversions

In [0]:
type_mapping = {
    "payment_sequential": "INT",
    "payment_installments": "INT",
    "payment_value": "DECIMAL(18,2)"
}

for column, target_type in type_mapping.items():

    failed_casts = bronze_payments.filter(
        F.col(column).isNotNull()
        & F.expr(
            f"try_cast(`{column}` AS {target_type})"
        ).isNull()
    ).count()

    print(f"{column}: {failed_casts} failed casts")

    assert failed_casts == 0, (
        f"Invalid conversion in {column}"
    )

# Check monetary precision before DECIMAL(18,2)
invalid_money = bronze_payments.filter(
    F.col("payment_value").isNotNull()
    & ~F.trim(F.col("payment_value")).rlike(
        r"^-?[0-9]+(\.[0-9]{1,2})?$"
    )
).count()

print("Invalid payment formats:", invalid_money)

assert invalid_money == 0

print("PAYMENT TYPE VALIDATION PASSED")

### Transform Payments into Silver
- Apply transformations

In [0]:
payments_clean = (
    bronze_payments

    .withColumn(
        "payment_sequential",
        F.expr("try_cast(payment_sequential AS INT)")
    )

    .withColumn(
        "payment_installments",
        F.expr("try_cast(payment_installments AS INT)")
    )

    .withColumn(
        "payment_value",
        F.expr("try_cast(payment_value AS DECIMAL(18,2))")
    )

    .withColumn(
        "payment_type",
        F.lower(F.trim(F.col("payment_type")))
    )

    .withColumn(
        "payment_value_quality",
        F.when(F.col("payment_value").isNull(), "MISSING")
        .when(F.col("payment_value") < 0, "NEGATIVE")
        .when(F.col("payment_value") == 0, "ZERO")
        .otherwise("POSITIVE")
    )

    .withColumn(
        "installment_quality",
        F.when(
            F.col("payment_installments").isNull(),
            "MISSING"
        )
        .when(
            F.col("payment_installments") < 0,
            "NEGATIVE"
        )
        .when(
            F.col("payment_installments") == 0,
            "ZERO"
        )
        .otherwise("POSITIVE")
    )

    .withColumn(
        "silver_processed_at",
        F.current_timestamp()
    )
)

payments_clean.printSchema()

### Validate composite primary keys
- Validate keys

In [0]:
source_count = bronze_payments.count()
silver_count = payments_clean.count()

null_keys = payments_clean.filter(
    F.col("order_id").isNull()
    | F.col("payment_sequential").isNull()
).count()

duplicate_keys = (
    payments_clean
    .groupBy("order_id", "payment_sequential")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

invalid_sequences = payments_clean.filter(
    F.col("payment_sequential") <= 0
).count()

print("Bronze rows:", source_count)
print("Silver rows:", silver_count)
print("Null composite keys:", null_keys)
print("Duplicate composite keys:", duplicate_keys)
print("Invalid payment sequences:", invalid_sequences)

assert source_count == silver_count == 103886
assert null_keys == 0
assert duplicate_keys == 0
assert invalid_sequences == 0

print("PAYMENT KEY VALIDATION PASSED")

### Financial and installment validation
- Examine payment-quality distributions

In [0]:
negative_payments = payments_clean.filter(
    F.col("payment_value") < 0
).count()

missing_payments = payments_clean.filter(
    F.col("payment_value").isNull()
).count()

print("Negative payments:", negative_payments)
print("Missing payment amounts:", missing_payments)

assert negative_payments == 0
assert missing_payments == 0

print("PAYMENT FINANCIAL VALIDATION PASSED")

### Validate the relationship with Orders
- Check orphan records

In [0]:
order_keys = (
    spark.table("olist_mlops.silver.orders_clean")
    .select("order_id")
    .distinct()
)

orphan_payments = (
    payments_clean
    .select("order_id")
    .distinct()
    .join(
        order_keys,
        on="order_id",
        how="left_anti"
    )
    .count()
)

print("Orphan payment order IDs:", orphan_payments)

assert orphan_payments == 0

print("PAYMENT REFERENTIAL INTEGRITY PASSED")

### Save the Silver table
- Write Delta table

In [0]:
(
    payments_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(SILVER_TABLE)
)

written_count = spark.table(SILVER_TABLE).count()

assert written_count == 103886

print("Created:", SILVER_TABLE)
print("Rows:", written_count)

### Analyze Payments using SQL
- Query A — Payment summary

In [0]:
%sql

SELECT
    COUNT(*) AS payment_records,
    COUNT(DISTINCT order_id) AS orders_with_payments,
    SUM(payment_value) AS total_payment_value,
    MIN(payment_value) AS minimum_payment,
    MAX(payment_value) AS maximum_payment,
    AVG(payment_value) AS average_payment
FROM olist_mlops.silver.payments_clean;

- Query B — Payment methods

In [0]:
%sql

SELECT
    payment_type,
    COUNT(*) AS payment_records,
    SUM(payment_value) AS total_payment_value,
    ROUND(AVG(payment_value), 2) AS average_payment_value
FROM olist_mlops.silver.payments_clean
GROUP BY payment_type
ORDER BY payment_records DESC;

- Query C — Orders with multiple payment records

In [0]:
%sql

SELECT
    order_id,
    COUNT(*) AS number_of_payment_records,
    COUNT(DISTINCT payment_type) AS payment_methods_used,
    SUM(payment_value) AS order_payment_total
FROM olist_mlops.silver.payments_clean
GROUP BY order_id
HAVING COUNT(*) > 1
ORDER BY number_of_payment_records DESC
LIMIT 10;